# Etapa 2 – Ingeniería de datos: Airbnb Clark County (Las Vegas)
Datos: Inside Airbnb, listings.csv.gz, fecha 2026-09-20

## 1. Carga de datos
Se carga el archivo de Inside Airbnb (Clark County, 2026-09-20): 20.651 anuncios y 90 variables.

In [12]:
import pandas as pd

df = pd.read_csv('listings.csv', low_memory=False)
print("Filas y columnas:", df.shape)

Filas y columnas: (20651, 90)


## 2. Tipos de variables
Hay 61 variables numéricas y 29 de texto. `price` está como texto por el "$" y debe convertirse a número.

In [13]:
print(df.dtypes.value_counts())
df.head()

float64    37
object     29
int64      24
Name: count, dtype: int64


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,44701,https://www.airbnb.com/rooms/44701,20260920135843,2026-09-23,NaN,"CES Jan3-10, 2027, 3 bed,2 baths, 1020sqft, Sl...",Experience the luxury of our well-appointed vi...,NaN,https://a0.muscache.com/pictures/1218626b-295f...,189245,...,5.00,4.75,4.50,Exempt,NaN,1,1,0,0,0.03
1,113019,https://www.airbnb.com/rooms/113019,20260920135843,2026-09-22,previous scrape,MGM Signature Studio !,"*3 night minimum, MAY accept two night stay on...",NaN,https://a0.muscache.com/pictures/56149560/d180...,575684,...,4.44,4.81,4.61,Exempt,NaN,6,5,1,0,1.12
2,114140,https://www.airbnb.com/rooms/114140,20260920135843,2026-09-22,NaN,MGM Signature 1BR/2BA Suite W/ View,"*3 night minimum, will accept 2 nights on occa...",NaN,https://a0.muscache.com/pictures/56068170/7fa6...,575684,...,4.41,4.78,4.63,Exempt,NaN,6,5,1,0,0.95
3,133084,https://www.airbnb.com/rooms/133084,20260920135843,2026-09-22,previous scrape,"Beautiful, Sleek, Sexy Condo - Long Term Rental","This immaculate, modern loft is available for ...",NaN,https://a0.muscache.com/pictures/30c3a88e-3c4b...,653641,...,5.00,4.50,5.00,NaN,NaN,1,1,0,0,0.01
4,143096,https://www.airbnb.com/rooms/143096,20260920135843,2026-09-23,NaN,"Las Vegas House with Private Pool 2,000 sq ft","Renting entire 2,000 square feet home, 3 bedro...",NaN,https://a0.muscache.com/pictures/b3c4bba8-ebb8...,694506,...,4.90,4.79,4.71,NaN,NaN,1,1,0,0,1.54


## 3. Datos faltantes
Se identifican las columnas con valores vacíos para decidir si se eliminan, se rellenan o se tratan después.

In [14]:
nulos = pd.DataFrame({
    'nulos': df.isnull().sum(),
    'porcentaje': (df.isnull().mean() * 100).round(2),
    'tipo': df.dtypes
}).sort_values('porcentaje', ascending=False)

nulos[nulos['nulos'] > 0]

,nulos,porcentaje,tipo
neighborhood_overview,20651,100.00,float64
host_since,20651,100.00,float64
host_response_time,20651,100.00,float64
host_thumbnail_url,20651,100.00,float64
host_acceptance_rate,20651,100.00,float64
host_response_rate,20651,100.00,float64
host_verifications,20651,100.00,float64
neighbourhood,20651,100.00,float64
host_total_listings_count,20651,100.00,float64
host_neighbourhood,20651,100.00,float64


### Análisis de datos faltantes

| Variable(s) | % nulos | Decisión | Justificación |
|---|---|---|---|
| 14 columnas (host_since, bathrooms, instant_bookable, etc.) | 100% | **Se eliminan** | No tienen ningún dato, no se pueden rellenar |
| price | 15.5% | **Se eliminan las filas** | Es la variable a predecir; no se debe inventar |
| review_scores_*, reviews_per_month, last_review | 29.5% | **Se dejan vacíos** | Son anuncios sin reseñas; rellenarlos introduce sesgo. Solo reseñas por mes pasa a 0 (es su valor real). Se crea `has_reviews` (paso 12) |
| beds, bedrooms, bathrooms, minimum/maximum_nights | < 3% | **Se eliminan las filas** | Son pocos; mejor borrar que inventar valores (paso 12) |
| bathrooms_text | 0.7% | **Se convierte** | Se usa para obtener el número de baños (paso 7) |
| license | 66% | **Se convierte** | Pasa a `has_license` e `is_exempt`; si está vacía, se deja vacía (paso 7) |
| source, host_about, host_location, first_review | 28–85% | **Se eliminan** | No son relevantes para el análisis (paso 5) |
| price_quote_*, estimated_revenue_l365d | 10–16% | **Se eliminan** | Se calculan a partir del precio (data leakage) (paso 5) |

**Dataset inicial:** 20.651 filas y 90 columnas (61 numéricas y 29 de texto).

## 4. Columnas vacías
Se eliminan 14 columnas sin ningún dato. Quedan 76.

In [15]:
# Columnas 100% vacías
cols_vacias = df.columns[df.isnull().all()].tolist()
print("Columnas vacías:", len(cols_vacias))

df = df.drop(columns=cols_vacias)
print("Nuevo shape:", df.shape)

Columnas vacías: 14
Nuevo shape: (20651, 76)


In [16]:
print(df.columns.tolist())

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_is_superhost', 'host_picture_url', 'host_listings_count', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availa

## 5. Selección de variables
Se conservan 34 variables útiles para predecir el precio y clasificar anuncios. Se eliminan identificadores, textos libres, variables redundantes y las calculadas a partir del precio (evitan data leakage).

In [17]:
columnas = [
    'id',
    # Anfitrión
    'hosts_time_as_host_years', 'host_is_superhost', 'host_listings_count',
    'host_has_profile_pic', 'host_identity_verified', 'calculated_host_listings_count',
    # Ubicación
    'neighbourhood_cleansed', 'latitude', 'longitude',
    # Propiedad
    'property_type', 'room_type', 'accommodates', 'bathrooms_text',
    'bedrooms', 'beds', 'amenities',
    # Precio y reservas
    'price', 'minimum_nights', 'maximum_nights',
    'availability_30', 'availability_365', 'estimated_occupancy_l365d',
    # Reseñas
    'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month', 'last_review',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',
    # Regulación
    'license'
]

df = df[columnas]
print("Nuevo shape:", df.shape)

Nuevo shape: (20651, 35)


### Selección de variables

**Se mantienen (33 + id):** características del anfitrión, ubicación, propiedad, precio, disponibilidad y reseñas, porque pueden influir en el precio o en si un anuncio es recomendable.

**Se eliminan:**
| Variables | Motivo |
|---|---|
| URLs, scrape_id, host_id, host_name, picture_url | Identificadores o enlaces; no aportan al análisis |
| name, description, host_about, host_location, source | Texto libre o metadatos del scraping |
| last_scraped, calendar_last_scraped | Fechas del scraping, no del anuncio |
| price_quote_*, estimated_revenue_l365d | Se calculan a partir del precio; usarlas para predecirlo sería trampa (data leakage) |
| minimum/maximum_*_nights, *_avg_ntm | Redundantes con minimum_nights y maximum_nights |
| availability_60/90/eoy, number_of_reviews_l30d/ly | Redundantes con otras variables de disponibilidad y reseñas |
| hosts_time_as_user_*, hosts_time_as_host_months | Redundantes con hosts_time_as_host_years |
| calculated_host_listings_count_* (por tipo) | Redundantes con calculated_host_listings_count |

**Nota:** `license` se convertirá en dos columnas (tiene número de licencia / está exento), relevante por la regulación de rentas cortas en Clark County. `last_review` se usará para calcular los días desde la última reseña.

## 6. Limpieza de precio
Se convierte `price` de texto a número (se quitan "$" y comas) y se eliminan los anuncios sin precio, ya que es la variable a predecir.

In [18]:
# Quitar "$" y "," y convertir a número
df['price'] = df['price'].str.replace('[$,]', '', regex=True).astype(float)

# Eliminar filas sin precio
df = df.dropna(subset=['price'])

print("Nuevo shape:", df.shape)
df['price'].describe()

Nuevo shape: (17456, 35)


,price
count,17456.000000
mean,324.055560
std,895.081123
min,5.000000
25%,133.000000
50%,229.635000
75%,361.165000
max,100040.000000


**Resultado:** quedan 17.456 anuncios. Precio mediano: \$229/noche. Se detectan valores extremos (mín. \$5, máx. \$100.040) que se tratarán en el análisis de outliers.

## 7. Limpieza de formatos
Se convierten variables de texto a formatos útiles: booleanos "t/f" a 1/0, baños a número, licencia a dos columnas (tiene número de licencia / está exento), última reseña a días transcurridos y amenities a cantidad.

Revisamos los valores de `license`: la mayoría dicen "Exempt" (exento), así que no es lo mismo que tener licencia.

In [19]:
pd.read_csv('listings.csv', low_memory=False)['license'].value_counts().head(10)

,count
license,
Exempt,5912
G12-34567,11
G63-07576,7
G63-07671,6
G63-07670,5
G63-07673,5
G12-36820,5
G73-01479,5
G63-07672,5


In [20]:
import json

# 1. Booleanos t/f → 1/0
for col in ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified']:
    df[col] = df[col].map({'t': 1, 'f': 0})

# 2. Baños: "1.5 baths" → 1.5 | "Half-bath" → 0.5 | compartido → columna aparte
df['bathrooms_shared'] = df['bathrooms_text'].str.contains('shared', case=False, na=False).astype(int)
df['bathrooms'] = df['bathrooms_text'].str.extract(r'(\d+\.?\d*)')[0].astype(float)
df.loc[df['bathrooms_text'].str.contains('half', case=False, na=False), 'bathrooms'] = 0.5
df = df.drop(columns='bathrooms_text')

# 3. Licencia → número real / exento; si está vacía, se deja vacía
lic = df['license']
df['is_exempt'] = (lic == 'Exempt').astype(float)
df['has_license'] = (lic.notnull() & (lic != 'Exempt')).astype(float)
df.loc[lic.isnull(), ['is_exempt', 'has_license']] = None
df = df.drop(columns='license')

# 4. Última reseña → días desde la última reseña (respecto a la fecha de los datos)
df['last_review'] = pd.to_datetime(df['last_review'])
df['days_since_last_review'] = (pd.Timestamp('2026-09-20') - df['last_review']).dt.days
df = df.drop(columns='last_review')

# 5. Amenities: lista en texto → cantidad de amenities
df['n_amenities'] = df['amenities'].apply(lambda x: len(json.loads(x)))

print("Nuevo shape:", df.shape)
df.head()

Nuevo shape: (17456, 38)


,id,hosts_time_as_host_years,host_is_superhost,host_listings_count,host_has_profile_pic,host_identity_verified,calculated_host_listings_count,neighbourhood_cleansed,latitude,longitude,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,bathrooms_shared,bathrooms,is_exempt,has_license,days_since_last_review,n_amenities
0,44701,15,0,3,1,1,1,Unincorporated Areas,36.11689,-115.16263,...,5.00,5.00,4.75,4.50,0,2.0,1.0,0.0,980.0,56
2,114140,15,0,8,1,1,6,Unincorporated Areas,36.10736,-115.16591,...,4.56,4.41,4.78,4.63,0,2.0,1.0,0.0,370.0,16
4,143096,15,1,1,1,1,1,Unincorporated Areas,36.05150,-115.22566,...,4.93,4.90,4.79,4.71,0,2.5,NaN,NaN,34.0,49
5,200680,15,0,11,1,1,10,Unincorporated Areas,36.10361,-115.29155,...,4.66,4.57,4.62,4.43,0,3.0,NaN,NaN,13.0,36
6,201619,15,0,2,1,1,1,Unincorporated Areas,36.07780,-115.27517,...,4.71,4.65,4.68,4.54,0,2.5,0.0,1.0,71.0,75


**Resultado:** se pasa de 35 a 38 columnas.
- Salen 3: `bathrooms_text`, `license`, `last_review` (reemplazadas por versiones numéricas).
- Entran 6: `bathrooms`, `bathrooms_shared`, `has_license`, `is_exempt`, `days_since_last_review`, `n_amenities`.
- En licencia, el 85% de los que tienen algo dicen "Exempt" (exento), por eso se separa de quienes tienen número de licencia real. Si la licencia estaba vacía, ambas quedan vacías.
- El índice queda salteado (0, 2, 4…) por las filas eliminadas; se reorganiza al final.

## 8. Variables categóricas
Se revisan las categorías de cada variable de texto para detectar inconsistencias y categorías con muy pocos registros.

In [21]:
for col in ['room_type', 'neighbourhood_cleansed', 'property_type']:
    print(f"\n--- {col}: {df[col].nunique()} categorías ---")
    print(df[col].value_counts())


--- room_type: 4 categorías ---
room_type
Entire home/apt    12429
Private room        4729
Hotel room           248
Shared room           50
Name: count, dtype: int64

--- neighbourhood_cleansed: 7 categorías ---
neighbourhood_cleansed
Unincorporated Areas       13544
City of Las Vegas           2119
City of North Las Vegas      800
City of Henderson            784
City of Mesquite             190
Boulder City                  15
Nellis AFB                     4
Name: count, dtype: int64

--- property_type: 54 categorías ---
property_type
Entire home                           4729
Entire rental unit                    2799
Entire condo                          2613
Room in hotel                         2194
Private room in home                  1614
Entire serviced apartment              880
Private room in resort                 715
Entire guesthouse                      375
Entire guest suite                     243
Entire villa                           180
Entire townhouse       

Las categorías están bien escritas, pero algunas tienen muy pocos anuncios. Juntamos Boulder City y Nellis AFB en "Other", y pasamos `property_type` de 54 a 6 grupos según el tipo de lugar (casa, apartamento, condo, etc.).

In [22]:
# Barrios con pocos anuncios → Other
df['neighbourhood_cleansed'] = df['neighbourhood_cleansed'].replace(
    ['Boulder City', 'Nellis AFB'], 'Other')

# Agrupar property_type por tipo de inmueble
def agrupar_propiedad(x):
    x = x.lower()
    if 'hotel' in x or 'resort' in x:
        return 'Hotel/Resort'
    elif 'condo' in x:
        return 'Condo'
    elif 'guest' in x:
        return 'Guesthouse'
    elif 'rental unit' in x or 'apartment' in x or 'loft' in x:
        return 'Apartment'
    elif any(p in x for p in ['home', 'townhouse', 'villa', 'bungalow', 'cottage', 'cabin']):
        return 'House'
    else:
        return 'Other'

df['property_type'] = df['property_type'].apply(agrupar_propiedad)

print(df['neighbourhood_cleansed'].value_counts())
print(df['property_type'].value_counts())

neighbourhood_cleansed
Unincorporated Areas       13544
City of Las Vegas           2119
City of North Las Vegas      800
City of Henderson            784
City of Mesquite             190
Other                         19
Name: count, dtype: int64
property_type
House           6918
Apartment       3865
Hotel/Resort    3158
Condo           2655
Guesthouse       665
Other            195
Name: count, dtype: int64


**Resultado:** quedan 6 zonas y 6 tipos de propiedad. La mayoría son casas (6.918) y casi todo está en zonas no incorporadas (13.544), que es donde está el Strip.

In [23]:
nulos = df.isnull().sum()
nulos[nulos > 0]

,0
bedrooms,9
beds,401
minimum_nights,118
maximum_nights,118
reviews_per_month,4724
review_scores_rating,4724
review_scores_accuracy,4724
review_scores_cleanliness,4724
review_scores_checkin,4725
review_scores_communication,4724


Revisamos los nulos que quedan después de la limpieza; se tratan en el paso 12.

## 9. Outliers
Revisamos las variables numéricas principales para encontrar valores extremos o imposibles.

In [24]:
cols = ['price', 'minimum_nights', 'maximum_nights', 'accommodates',
        'bedrooms', 'beds', 'bathrooms', 'n_amenities']

df[cols].describe(percentiles=[.01, .25, .5, .75, .99]).round(1)

,price,minimum_nights,maximum_nights,accommodates,bedrooms,beds,bathrooms,n_amenities
count,17456.0,17338.0,1.733800e+04,17456.0,17447.0,17055.0,17376.0,17456.0
mean,324.1,7.6,1.858413e+06,5.7,2.0,3.1,1.7,40.5
std,895.1,15.2,6.313940e+07,3.6,1.6,2.4,1.0,17.7
min,5.0,1.0,1.000000e+00,1.0,0.0,1.0,0.0,0.0
1%,28.0,1.0,5.000000e+00,1.0,0.0,1.0,1.0,7.0
25%,133.0,1.0,3.640000e+02,3.0,1.0,2.0,1.0,26.0
50%,229.6,2.0,3.650000e+02,4.0,2.0,2.0,1.5,41.0
75%,361.2,3.0,1.125000e+03,8.0,3.0,4.0,2.0,53.0
99%,1765.0,31.0,1.125000e+03,16.0,7.0,12.0,5.0,81.0
max,100040.0,365.0,2.147484e+09,16.0,16.0,27.0,14.5,112.0


| Variable | Qué se ve | Decisión |
|---|---|---|
| price | Máx. \$100.040 vs. 99% en \$1.765; mín. \$5 no es real | Borrar fuera del 1%–99% (\$28 a \$1.765) |
| minimum_nights | 99% hasta 31 noches, máx. 365 | Borrar > 31 (arriendos largos) |
| maximum_nights | Máx. de 2.147 millones (valor del sistema); 1.125 = "sin límite" | Topar en 1.125 |
| beds | Hasta 27 camas, pero máx. 16 huéspedes | Borrar > 16 |
| accommodates, bedrooms, bathrooms, n_amenities | Valores posibles | Se dejan igual |

Quitamos los precios muy bajos o muy altos (fuera del 1% y 99%: \$28 a \$1.765), los anuncios de más de 31 noches mínimas (son arriendos largos) y los de más de 16 camas (más que el máximo de huéspedes). En noches máximas, los valores mayores a 1.125 se cambian por 1.125, que en Airbnb significa "sin límite". El resto de variables tienen valores posibles y se dejan igual.

In [25]:
filas_antes = len(df)

# Precio: entre percentil 1 y 99
p1, p99 = df['price'].quantile([0.01, 0.99])
df = df[(df['price'] >= p1) & (df['price'] <= p99)]

# Noches mínimas y camas (se conservan los nulos, se tratan después)
df = df[~(df['minimum_nights'] > 31)]
df = df[~(df['beds'] > 16)]

# Noches máximas: topar en 1125
df['maximum_nights'] = df['maximum_nights'].clip(upper=1125)

print(f"Filas eliminadas: {filas_antes - len(df)} ({(filas_antes - len(df))/filas_antes:.1%})")
print("Shape:", df.shape)
df['price'].describe().round(1)

Filas eliminadas: 517 (3.0%)
Shape: (16939, 38)


,price
count,16939.0
mean,287.6
std,234.3
min,28.0
25%,135.5
50%,230.5
75%,358.5
max,1765.0


**Resultado:** se eliminaron 517 anuncios (3%). Quedan 16.939. El precio ahora va de \$28 a \$1.765, con mediana de \$230. La desviación estándar bajó de 895 a 234.

## 10. Amenities
Revisamos cuáles son las amenities más comunes para crear columnas sí/no con las más relevantes.

In [26]:
from collections import Counter

conteo = Counter()
for lista in df['amenities']:
    conteo.update(json.loads(lista))

pd.DataFrame(conteo.most_common(40), columns=['amenity', 'anuncios'])

,amenity,anuncios
0,Wifi,16155
1,Smoke alarm,16013
2,Kitchen,14584
3,Hair dryer,13620
4,Hot water,13456
5,Air conditioning,13443
6,Carbon monoxide alarm,13012
7,Iron,12939
8,Free parking on premises,12801
9,Hangers,12745


Elegimos 10 comodidades que diferencian a los anuncios y pueden influir en el precio (piscina, jacuzzi, parqueadero, cocina, etc.). No se usan las que tienen casi todos, como wifi o alarma de humo, porque no marcan diferencia. Después se elimina la columna original y se conserva el total de comodidades (`n_amenities`).

In [27]:
amenities_clave = {
    'has_pool':        r'pool(?! table)',      # piscina, pero no "mesa de pool"
    'has_hot_tub':     r'hot tub',
    'has_parking':     r'free parking',
    'has_kitchen':     r'kitchen',
    'has_washer':      r'(?<!dish)washer',     # lavadora, pero no lavaplatos
    'has_ac':          r'air conditioning',
    'has_workspace':   r'workspace',
    'has_self_checkin':r'self check-in',
    'has_gym':         r'gym',
    'has_pets':        r'pets allowed'
}

texto = df['amenities'].str.lower()
for col, patron in amenities_clave.items():
    df[col] = texto.str.contains(patron, regex=True).astype(int)

df = df.drop(columns='amenities')

print("Shape:", df.shape)
df[list(amenities_clave)].mean().round(2).sort_values(ascending=False)

Shape: (16939, 47)


,0
has_ac,0.97
has_kitchen,0.89
has_parking,0.80
has_washer,0.74
has_self_checkin,0.70
has_pool,0.68
has_workspace,0.64
has_hot_tub,0.49
has_gym,0.38
has_pets,0.24


In [28]:
df = df.drop(columns='has_ac')
print("Shape:", df.shape)

Shape: (16939, 46)


**Resultado:** quedan 9 columnas de comodidades. Se quitó aire acondicionado porque lo tiene el 97% de los anuncios y no marca diferencia. Las más comunes son cocina (89%) y parqueadero (80%); las menos, gimnasio (38%) y mascotas (24%).

## 11. Codificación de variables categóricas
Los modelos solo entienden números, así que convertimos las variables de texto en columnas de 1 y 0 (una por categoría).

In [29]:
df = pd.get_dummies(df, columns=['room_type', 'property_type', 'neighbourhood_cleansed'],
                    drop_first=True, dtype=int)

print("Shape:", df.shape)
print("Columnas de texto restantes:", df.select_dtypes('object').columns.tolist())

Shape: (16939, 56)
Columnas de texto restantes: []


**Resultado:** las 3 variables categóricas se convirtieron en 13 columnas de 1 y 0. Quedan 56 columnas, todas numéricas.

## 12. Datos faltantes
Camas, baños, habitaciones y noches tienen pocos nulos, así que borramos esas filas. En reseñas por mes ponemos 0, porque sin reseñas ese es el valor real. Las calificaciones, los días desde la última reseña y la licencia se dejan vacíos para no introducir sesgos (recomendación del profesor). Creamos `has_reviews` para identificar los anuncios sin reseñas. Al hacer modelos con esas variables se usarán solo las filas con valor.

In [30]:
# 1. Marcar anuncios sin reseñas
df['has_reviews'] = df['review_scores_rating'].notnull().astype(int)

# 2. Pocos nulos → borrar filas
df = df.dropna(subset=['bedrooms', 'beds', 'bathrooms', 'minimum_nights', 'maximum_nights'])

# 3. Reseñas por mes: sin reseñas = 0 (es un dato real, no inventado)
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# 4. Calificaciones, días desde la última reseña y licencia: se dejan vacíos
nulos = df.isnull().sum()
print(nulos[nulos > 0])
print("Shape:", df.shape)

review_scores_rating            4338
review_scores_accuracy          4338
review_scores_cleanliness       4338
review_scores_checkin           4339
review_scores_communication     4338
review_scores_location          4339
review_scores_value             4339
is_exempt                      10576
has_license                    10576
days_since_last_review          4338
dtype: int64
Shape: (16434, 57)


**Resultado:** se eliminaron 505 anuncios (3%). Quedan 16.434 anuncios y 57 columnas. Solo quedan vacíos en calificaciones, días desde la última reseña y licencia, a propósito, para no inventar datos.

## 13. Duplicados
Revisamos si hay anuncios repetidos.

In [31]:
print("Filas duplicadas:", df.duplicated().sum())
print("IDs duplicados:", df['id'].duplicated().sum())

Filas duplicadas: 0
IDs duplicados: 0


**Resultado:** no hay anuncios repetidos.

## 14. Dataset final
Reorganizamos el índice y guardamos el dataset limpio para las siguientes etapas (análisis exploratorio y modelos).

In [32]:
# Reorganizar índice (0, 1, 2, ...)
df = df.reset_index(drop=True)

# Resumen final
print("Shape final:", df.shape)
print("Nulos:", df.isnull().sum().sum())
print("Columnas de texto:", df.select_dtypes('object').shape[1])

# Guardar y descargar
df.to_csv('listings_clark_limpio.csv', index=False)

from google.colab import files
files.download('listings_clark_limpio.csv')

Shape final: (16434, 57)
Nulos: 55859
Columnas de texto: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>